In [ ]:
%pip install transformers datasets torch accelerate evaluate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 11.8 MB/s eta 0:00:00


In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

In [ ]:
model_name="distilbert/distilbert-base-uncased"
tokenizer=DistilBertTokenizer.from_pretrained(model_name)
model=DistilBertForSequenceClassification.from_pretrained(model_name,num_labels=3)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
inputs=["This product exceeded all my expectations and works perfectly!",
"The battery life is amazing, and the phone feels great in my hand.",
"I'm so happy with their customer service—they solved my issue quickly.",
"The food was absolutely delicious; I would highly recommend this restaurant.",
"The item broke after one use, and the support team was completely unhelpful.",
"I am amazed by the speed of the processor but disappointed that it heats up quickly.",
"This was the worst movie I've ever seen. What a terrible waste of money.",
"The delivery was late, and the package arrived damaged."]

In [ ]:
def tokenize(example):
  # Process the batch of texts in the 'English' column
  return tokenizer(example['English'], truncation=True, max_length=100, padding='max_length', padding_side='left', return_tensors="pt")

In [ ]:
from datasets import load_dataset,Dataset
import pandas as pd

In [ ]:
data=load_dataset('leduckhai/Sentiment-Reasoning')

In [ ]:
training_dataset=data['train'].remove_columns([col for col in data['train'].column_names if col not in ['English', 'label']])
validation_dataset=data['test'].remove_columns([col for col in data['test'].column_names if col not in ['English', 'label']])

In [ ]:
from transformers import Trainer, TrainingArguments
import evaluate

In [ ]:
# training
labels={'neutral':0,'positive':1,'negative':2}
tokenize_datasets=training_dataset.map(tokenize,batched=True,remove_columns=['English'])
tokenize_datasets=tokenize_datasets.map(lambda example: {'label': labels[example['label']]})
tokenize_datasets=tokenize_datasets.rename_column('label','labels')
tokenize_datasets.set_format("torch")

# val
tokenize_validation_dataset = validation_dataset.map(tokenize, batched=True, remove_columns=['English'])
tokenize_validation_dataset=tokenize_validation_dataset.map(lambda example: {'label': labels[example['label']]})
tokenize_validation_dataset=tokenize_validation_dataset.rename_column('label','labels')
tokenize_validation_dataset.set_format("torch")

In [ ]:
tokenize_datasets

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 5612
})

In [ ]:
accuracy=evaluate.load('accuracy')
f1=evaluate.load('f1')

def compute_metrice(eval_metrice):
  logits,labels=eval_metrice
  predictions=torch.argmax(torch.from_numpy(logits),dim=-1)
  acc = accuracy.compute(predictions=predictions, references=labels)['accuracy']
  f1_score = f1.compute(predictions=predictions, references=labels, average='weighted')['f1']
  return {"accuracy": acc, "f1": f1_score}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=2e-4,
    eval_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",        # Save checkpoints at the end of each epoch
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model.to('cuda'),
    args=training_args,
    train_dataset=tokenize_datasets,  # Assumes pre-processed dataset
    eval_dataset=tokenize_validation_dataset,    # Assumes pre-processed dataset
    compute_metrics=compute_metrice, # Function to compute metrics
)

In [ ]:
trainer.train()

wandb: Currently logged in as: medicalassistance-ai (medicalassistance-ai-maai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.978196,0.624198,0.622883
2,No log,1.006318,0.613199,0.608127
3,No log,1.444796,0.610449,0.608990
4,No log,1.685919,0.611366,0.607145
5,No log,1.740781,0.612741,0.609974
6,0.288200,2.210823,0.614574,0.613890
7,0.288200,2.613749,0.620532,0.618254
8,0.288200,2.909141,0.613657,0.607515
9,0.288200,2.894205,0.616407,0.613936
10,0.288200,2.920251,0.618698,0.616207


TrainOutput(global_step=880, training_loss=0.16887943907217545, metrics={'train_runtime': 700.8918, 'train_samples_per_second': 80.069, 'train_steps_per_second': 1.256, 'total_flos': 1451992771224000.0, 'train_loss': 0.16887943907217545, 'epoch': 10.0})

In [ ]:
lil_input=[]
for inp in inputs:
  tokens=tokenizer(inp, truncation=True, max_length=100, padding='max_length', padding_side='left', return_tensors="pt").to('cuda')
  print(tokens)
  lil_input.append(tokens)

{'input_ids': tensor([[    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,   101,  2023,
          4031, 14872,  2035,  2026, 10908,  1998,  2573,  6669,   999,   102]],
       device='cuda:0'), 'attention_mask': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
for out in lil_input:
  with torch.no_grad():
    output = model(**out)
    # Get the predicted class index from the logits
    predicted_class_index = torch.argmax(output.logits, dim=-1)
    print(f"-> Predicted class index: {predicted_class_index.item()}")

-> Predicted class index: 1
-> Predicted class index: 1
-> Predicted class index: 1
-> Predicted class index: 1
-> Predicted class index: 2
-> Predicted class index: 2
-> Predicted class index: 2
-> Predicted class index: 2


## Lets try peft now

In [ ]:
from peft import LoraConfig,get_peft_model,TaskType,prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
model.config.use_cache=False

In [ ]:
model=prepare_model_for_kbit_training(model)

In [ ]:
Loraconfig=LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_lin','v_lin'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_CLS
)

In [ ]:
peft_model=get_peft_model(model,Loraconfig)

In [ ]:
peft_model.print_trainable_parameters()

trainable params: 887,811 || all params: 67,843,590 || trainable%: 1.3086


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=2e-4,
    eval_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",        # Save checkpoints at the end of each epoch
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=peft_model.to('cuda'),
    args=training_args,
    train_dataset=tokenize_datasets,  # Assumes pre-processed dataset
    eval_dataset=tokenize_validation_dataset,    # Assumes pre-processed dataset
    compute_metrics=compute_metrice, # Function to compute metrics
)

In [ ]:
trainer.train()

wandb: Currently logged in as: medicalassistance-ai (medicalassistance-ai-maai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.937365,0.610907,0.605263
2,No log,0.911512,0.627864,0.625036
3,No log,0.917634,0.632447,0.628316
4,No log,0.960184,0.630614,0.626150
5,No log,0.922205,0.638405,0.636297
6,0.570000,0.942601,0.637489,0.634820
7,0.570000,0.982599,0.636114,0.633968
8,0.570000,0.969572,0.641613,0.640493
9,0.570000,0.998655,0.636572,0.634481
10,0.570000,0.996064,0.637489,0.635998


TrainOutput(global_step=880, training_loss=0.5181391629305753, metrics={'train_runtime': 384.679, 'train_samples_per_second': 145.888, 'train_steps_per_second': 2.288, 'total_flos': 1481887143216000.0, 'train_loss': 0.5181391629305753, 'epoch': 10.0})

In [ ]:
lil_input=[]
#tokenizer.pad_token = tokenizer.eos_token
for inp in inputs:
  tokens=tokenizer(inp, truncation=True, max_length=100, padding='max_length', padding_side='left', return_tensors="pt").to('cuda')
  print(tokens)
  lil_input.append(tokens)

{'input_ids': tensor([[    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,   101,  2023,
          4031, 14872,  2035,  2026, 10908,  1998,  2573,  6669,   999,   102]],
       device='cuda:0'), 'attention_mask': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
for out in lil_input:
  with torch.no_grad():
    output = model(**out)
    # Get the predicted class index from the logits
    predicted_class_index = torch.argmax(output.logits, dim=-1)
    print(f"-> Predicted class index: {predicted_class_index.item()}")

-> Predicted class index: 1
-> Predicted class index: 1
-> Predicted class index: 1
-> Predicted class index: 1
-> Predicted class index: 2
-> Predicted class index: 0
-> Predicted class index: 2
-> Predicted class index: 2


as you can see even thoough performence have reduced but training time has halfed

## Project 2: Domain-Specific Summarization

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 15.9 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer,BitsAndBytesConfig,TextStreamer
import evaluate

In [ ]:
dataset=load_dataset('ccdv/pubmed-summarization')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00005.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

section/train-00001-of-00005.parquet:   0%|          | 0.00/208M [00:00<?, ?B/s]

section/train-00002-of-00005.parquet:   0%|          | 0.00/207M [00:00<?, ?B/s]

...

section/validation-00000-of-00001.parque(…):   0%|          | 0.00/59.0M [00:00<?, ?B/s]

section/test-00000-of-00001.parquet:   0%|          | 0.00/58.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

In [ ]:
# Split the 'train' split of the dataset
dataset1 = dataset['train'].train_test_split(test_size=0.1)

In [ ]:
model_name='facebook/bart-large-cnn'
tokenize=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name,device_map='auto')

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 119924
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6633
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6658
    })
})

In [ ]:
dataset1

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 107931
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 11993
    })
})

In [ ]:
x=dataset['train']['abstract']

In [ ]:
x

Column(["background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and 2009 on 2897 primary and secondary school boys and girls ( 7 - 13 years old ) based on advocacy approach in shiraz , iran . \n the project provided nutritious snacks in public schools over a 2-year period along with advocacy oriented actions in order to implement and promote nutritional intervention . for evaluation of effectiveness of the intervention growth monitoring indices of pre- and post - intervention were statistically compared.results:the frequency of subjects with body mass index lower than 5% decreased significantly after intervention among girls ( p = 0.02 ) . \n however , there were no significant changes among boys or total population . \n the mean of all anthropomet

In [ ]:
def tokenizer(example):
  return tokenize(example['text'],max_length=1024,truncation=True,padding_side='left',padding='max_length',return_tensors='pt')

In [ ]:
def text_gen(example):
  example['text']=f"Summarize this medical abstract: {example['article']}\nSummary: {example['abstract']}"
  return example

In [ ]:
data=dataset1.map(text_gen)

NameError: name 'dataset1' is not defined

In [ ]:
data

Dataset({
    features: ['article', 'abstract', 'text'],
    num_rows: 107931
})

In [ ]:
tokenized_train=data.map(tokenizer,batched=True,remove_columns=["article", "abstract", "text"])

Map:   0%|          | 0/107931 [00:00<?, ? examples/s]

In [ ]:
tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 107931
})

In [ ]:
print(tokenized_train[0])

{'input_ids': [0, 38182, 3916, 2072, 42, 1131, 20372, 35, 11, 189, 11735, 2156, 5, 474, 575, 5200, 942, 36, 1368, 19911, 102, 4839, 1027, 10, 2570, 7, 14518, 812, 111, 1330, 1042, 88, 5, 26467, 1322, 12429, 3207, 467, 36, 752, 5124, 2156, 11735, 102, 4839, 479, 1437, 50118, 1165, 11, 5, 2570, 21, 41, 1965, 7, 9160, 5, 1850, 4460, 812, 36, 1663, 701, 4839, 22507, 467, 13, 4566, 18381, 21875, 11, 5, 701, 9, 1098, 1663, 479, 1437, 50118, 1368, 19911, 102, 128, 29, 2570, 21, 45, 6264, 142, 2156, 19, 5, 32442, 27587, 1229, 13229, 1760, 9, 11735, 2156, 12442, 3148, 124, 5, 864, 9, 217, 812, 11, 5, 12429, 3207, 467, 36, 181, 3275, 4839, 454, 9633, 1437, 50118, 479, 959, 2156, 5, 414, 1542, 341, 30, 1368, 19911, 102, 7, 37357, 5, 181, 3275, 1663, 701, 1965, 16, 9, 4566, 773, 13, 1966, 9, 474, 575, 2122, 1663, 3926, 479, 1437, 50118, 71, 9311, 10, 346, 9, 801, 414, 1715, 2156, 1368, 19911, 102, 3919, 5, 32633, 1589, 414, 1915, 14948, 36, 32633, 1589, 13911, 4839, 1663, 801, 29, 414, 1542, 25, 5

In [ ]:
model.save_pretrained('bart_model')
tokenize.save_pretrained('bart_model')

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('bart_model/tokenizer_config.json',
 'bart_model/special_tokens_map.json',
 'bart_model/vocab.json',
 'bart_model/merges.txt',
 'bart_model/added_tokens.json',
 'bart_model/tokenizer.json')

In [ ]:
from transformers import pipeline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bart_model')
input_text = f"Summarize this medical report: {x}"  # Replace with your actual x
tokens = tokenize(input_text)

In [ ]:
print(f"Tokenized length: {len(tokens['input_ids'])}")  # If >1024, length is the problem
print(f"Max token ID: {max(tokens['input_ids'])}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

Tokenized length: 1481
Max token ID: 49329
Tokenizer vocab size: 50265


In [ ]:
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)  # Explicit device

Device set to use cpu


In [ ]:
x[0].split('.')

['background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran',
 'materials and methods : this case - control nutritional intervention has been done between 2008 and 2009 on 2897 primary and secondary school boys and girls ( 7 - 13 years old ) based on advocacy approach in shiraz , iran ',
 ' \n the project provided nutritious snacks in public schools over a 2-year period along with advocacy oriented actions in order to implement and promote nutritional intervention ',
 ' for evaluation of effectiveness of the intervention growth monitoring indices of pre- and post - intervention were statistically compared',
 'results:the frequency of subjects with body mass index lower than 5% decreased significantly after intervention among girls ( p = 0',
 '02 ) ',
 ' \n however , there were no significant changes among boys or total population ',
 ' \n the me

In [ ]:
text_summery=[]
for i in x[0].split('.'):
  input_text = f"Summarize this medical report: {i}"
  y = summarizer(input_text, max_length=30, min_length=10, do_sample=False)
  text_summery.append(y)

Your max_length is set to 30, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 30, but your input_length is only 25. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)
Your max_length is set to 30, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 30, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_le

In [ ]:
text_summery

[[{'summary_text': 'Study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shir'}],
 [{'summary_text': 'This case - control nutritional intervention has been done between 2008 and 2009 on 2897 primary and secondary school boys and girls ( 7 - 13'}],
 [{'summary_text': 'The project provided nutritious snacks in public schools over a 2-year period.'}],
 [{'summary_text': 'Growth monitoring indices of pre- and post - intervention were statistically compared.'}],
 [{'summary_text': 'The frequency of subjects with body mass index lower than 5% decreased significantly after intervention among girls ( p = 0)'}],
 [{'summary_text': 'CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery. Visit CNN.com/Travel each week for a'}],
 [{'summary_text': 'There were no significant changes among boys or total population.'}],
 [{'summary_text': 'The mean of all anthropometric in

In [ ]:
complete_text=' '.join(text[0]['summary_text'] for text in text_summery)

In [ ]:
input_text = f"Summarize this medical report: {complete_text}"
y = summarizer(input_text, max_length=200, min_length=50, do_sample=False)

In [ ]:
y

[{'summary_text': 'Study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shir. Project provided nutritious snacks in public schools over a 2-year period. Study demonstrates the potential success and scalability of school feeding programs in iran.'}]

In [ ]:
text_to_split=dataset['train']['article'][0]

In [ ]:
import re

re.split()

In [ ]:
import re
from transformers import AutoTokenizer  # Assume you have this for token counting

def recursive_chunk(text, max_tokens=512, overlap_ratio=0.2, tokenizer=tokenize):
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # Or your model's tokenizer

    # Define separators in order of hierarchy: paragraphs > sentences > words
    separators = ['\n\n', '\n', '. ', ' ']  # Customize for your text (e.g., add section markers like '##')

    def split_with_hierarchy(current_text, sep_index=0):
        if sep_index >= len(separators):
            return [current_text]  # Base case: can't split further

        chunks = []
        parts = re.split(separators[sep_index], current_text)
        current_chunk = ""
        for part in parts:
            temp_chunk = current_chunk + (separators[sep_index] if current_chunk else "") + part
            token_count = len(tokenizer.encode(temp_chunk))
            if token_count > max_tokens:
                # If too big, recurse to finer separator
                sub_chunks = split_with_hierarchy(temp_chunk, sep_index + 1)
                chunks.extend(sub_chunks[:-1])  # Add all but last
                current_chunk = sub_chunks[-1]  # Carry over last sub-chunk
            else:
                current_chunk = temp_chunk

        if current_chunk:
            chunks.append(current_chunk)

        # Add overlap to chunks (except first)
        overlap_tokens = int(max_tokens * overlap_ratio)
        for i in range(1, len(chunks)):
            prev_tokens = tokenizer.encode(chunks[i-1])[-overlap_tokens:]
            overlap_text = tokenizer.decode(prev_tokens)
            chunks[i] = overlap_text + chunks[i]

        return chunks

    return split_with_hierarchy(text)

In [ ]:
chunk=recursive_chunk(text_to_split)

## project 4

In [ ]:
%pip install transformers datasets torch accelerate evaluate peft trl rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 5.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1780252ec151076f333364325dc07ac7b3ec7823a927f4db223e546d13ada9d6
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 16.5 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from peft import PromptTuningConfig,get_peft_model,PromptTuningInit, TaskType
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,Seq2SeqTrainer,Seq2SeqTrainingArguments
import torch
import evaluate

In [ ]:
dataset = load_dataset("gaussalgo/Canard_Wiki-augmented", split="train[:1000]")  # Use full for real
dataset = dataset.train_test_split(test_size=0.1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts'],
        num_rows: 900
    })
    test: Dataset({
        features: ['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts'],
        num_rows: 100
    })
})

In [ ]:
def text_generation(example):
  input_texts = []
  target_texts = []
  for i in range(len(example['History'])):
    hist = example['History'][i]
    '''if len(hist) != 0:
      hist_text = "\n".join(str(x) for x in hist)
    else:
      hist_text = 'NO HISTORY'''

    ques = example['Question'][i]

    input_texts.append(f"Rewrite query: {ques}")
    target_texts.append(example['Rewrite'][i])
  example['input_text'] = input_texts
  example['target_text'] = target_texts
  return example

In [ ]:
dataset_re=dataset.map(text_generation,batched=True,remove_columns=['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts'])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
model_name = "t5-small"  # Or "t5-base" for better performance
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name,device_map='auto')

In [ ]:
def tokenize(example):
  model_inputs=tokenizer(example['input_text'],max_length=512,truncation=True,padding='max_length')
  labels=tokenizer(example['target_text'],max_length=512,truncation=True,padding='max_length')
  model_inputs['labels']=labels['input_ids']
  return model_inputs

In [ ]:
dataset_ready=dataset_re.map(tokenize,batched=True,remove_columns=['input_text','target_text'])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
peft_config=PromptTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,  # Or RANDOM
    num_virtual_tokens=30,  # Soft prompt length
    prompt_tuning_init_text="Rewrite the query:",  # Optional init text
    tokenizer_name_or_path=model_name
)

In [ ]:
model=get_peft_model(model,peft_config)

In [ ]:
bleu=evaluate.load('bleu')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["bleu"]}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./p_tuned_query_rewriter",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    #gradient_accumulation_steps=4,
    fp16=True,
    learning_rate=5e-4,  # Higher for prompts
    #weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    #report_to="none",
    #predict_with_generate=True
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_ready["train"],
    eval_dataset=dataset_ready["test"],
    compute_metrics=compute_metrics
    #tokenizer=tokenizer
)

# Train and save
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
wandb: Currently logged in as: medicalassistance-ai (medicalassistance-ai-maai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss


In [ ]:
model.save_pretrained("./p_tuned_query_rewriter")
tokenizer.save_pretrained("./p_tuned_query_rewriter")

('./p_tuned_query_rewriter/tokenizer_config.json',
 './p_tuned_query_rewriter/special_tokens_map.json',
 './p_tuned_query_rewriter/spiece.model',
 './p_tuned_query_rewriter/added_tokens.json',
 './p_tuned_query_rewriter/tokenizer.json')

In [ ]:
input="when is it established?"
tokens=tokenizer(input,return_tensors='pt').to('cuda')

In [ ]:
from peft import PeftModelForCausalLM
from transformers import pipeline

In [ ]:
base_model=AutoModelForSeq2SeqLM.from_pretrained(model_name,device_map='auto')
peft_model=PeftModelForCausalLM.from_pretrained(base_model,'./p_tuned_query_rewriter')
rewiter=pipeline('text2text-generation',model=peft_model,tokenizer=tokenizer)
rewriter1=pipeline('text2text-generation',model=base_model,tokenizer=tokenizer)

Device set to use cuda:0
Device set to use cuda:0


In [ ]:
query = "when are you going to NY?"
rewritten = rewiter(f"Rewrite query: {query}", max_length=512)[0]["generated_text"]
print(rewritten)

Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


: when are you going to NY?


In [ ]:
rewiter(f"Rewrite query: {query}", max_length=512)

Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': ': when are you going to NY?'}]

In [ ]:
# Importing classes and functions from the transformers library
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Importing the load_dataset function from the datasets library
from datasets import load_dataset

# Importing specific classes and functions from the peft library
from peft import get_peft_model, PromptTuningConfig, TaskType, PromptTuningInit, PeftModel, PeftConfig

# Importing the notebook_login function from the huggingface_hub library
from huggingface_hub import notebook_login

# Importing the os and time modules
import os
import time

In [ ]:
# Specify the pre-trained model name you want to use
model_name = "bigscience/bloomz-560m"

# Load the tokenizer associated with the pre-trained model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pre-trained causal language model using the specified model name
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,device_map='auto')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [ ]:
# Tokenize the input text using the specified tokenizer
input1 = tokenizer("Two things are infinite: ", return_tensors="pt", padding=True).to('cuda')

# Generate text using the pre-trained foundation model based on the provided input_ids and attention_mask.
foundation_outputs = foundation_model.generate(
    input_ids=input1["input_ids"],
    attention_mask=input1["attention_mask"],
    max_new_tokens=7,
    eos_token_id=tokenizer.eos_token_id
)

# Decode the generated token IDs into human-readable text.
decoded_output = tokenizer.batch_decode(foundation_outputs, skip_special_tokens=True)

# Print the decoded output, which represents the generated text.
print(decoded_output)

['Two things are infinite:  the number of people and the number']


In [ ]:
# Load the "english_quotes" dataset using the load_dataset function from the datasets library
data = load_dataset("Abirate/english_quotes")

# Tokenize the quotes in the dataset using the specified tokenizer
data = data.map(lambda samples: tokenizer(samples["quote"]).to('cuda'), batched=True)

# Select a subset of the training samples (first 50 samples in this case)
train_sample = data["train"].select(range(50))

# Display the selected subset of training samples
display(train_sample)

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 50
})

In [ ]:
# Create a configuration for prompt tuning using the PromptTuningConfig class
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=4,
    tokenizer_name_or_path=model_name
)

# Get a PeftModel using the specified foundation_model and prompt tuning configuration
peft_model = get_peft_model(foundation_model, peft_config)

# Print the trainable parameters of the PeftModel
print(peft_model.print_trainable_parameters())

trainable params: 4,096 || all params: 559,218,688 || trainable%: 0.0007
None


In [ ]:
%mkdir /content/working_dir

In [ ]:
# Define the output directory for storing Peft model outputs
output_directory = os.path.join("/content/working_dir", "peft_outputs")

# Create the working directory if it doesn't exist
if not os.path.exists("/content/working_dir"):
    os.mkdir("/content/working_dir")

# Create the output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.mkdir(output_directory)

# Define training arguments for the Peft model
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    output_dir=output_directory,  # Where the model predictions and checkpoints will be written
    no_cuda=True,  # This is necessary for CPU clusters.
    auto_find_batch_size=True,  # Find a suitable batch size that will fit into memory automatically
    learning_rate=3e-2,  # Higher learning rate than full fine-tuning
    num_train_epochs=5  # Number of passes to go through the entire fine-tuning dataset
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [ ]:
# Enable gradient checkpointing in the Peft model's configuration
peft_model.config.gradient_checkpointing = True

# Create a Trainer instance for training the Peft model
trainer = Trainer(
    model=peft_model,  # We pass in the PEFT version of the foundation model, bloomz-560M
    args=training_args,  # Training arguments specifying output directory, GPU usage, batch size, etc.
    train_dataset=train_sample,  # Training dataset
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)  # mlm=False indicates not to use masked language modeling
)

# Start the training process
trainer.train()

wandb: Currently logged in as: medicalassistance-ai (medicalassistance-ai-maai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


In [ ]:
from datasets import load_dataset
from peft import PromptTuningConfig, get_peft_model, PromptTuningInit, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,  # Use this for seq2seq specifics
    Seq2SeqTrainingArguments,  # Better for generation args
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
import evaluate
import torch
import os
import numpy as np

# Memory env
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# Load dataset
dataset = load_dataset("gaussalgo/Canard_Wiki-augmented")

# Subset for quick testing (remove after debugging)
dataset = dataset.map(lambda x: x, remove_columns=dataset['train'].column_names)  # Placeholder
dataset['train'] = dataset['train'].select(range(1000))  # Test on 1000 samples
dataset['test'] = dataset['test'].select(range(200))     # Small eval

def text_generation(example):
    input_texts = []
    target_texts = []
    for i in range(len(example['Question'])):
        ques = example['Question'][i]
        input_texts.append(f"Rewrite query: {ques}")
        target_texts.append(example['Rewrite'][i])
    example['input_text'] = input_texts
    example['target_text'] = target_texts
    return example

dataset = dataset.map(text_generation, batched=True)

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Quantization (keep for memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16
)

def tokenize(example):
    model_inputs = tokenizer(
        example['input_text'],
        max_length=256,
        truncation=True
    )
    labels = tokenizer(
        example['target_text'],
        max_length=128,
        truncation=True
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_datasets = dataset.map(
    tokenize,
    batched=True,
    remove_columns=[col for col in dataset['train'].column_names if col not in ['input_ids', 'attention_mask', 'labels']]
)

# P-Tuning (use RANDOM to avoid text echo)
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,  # Less prone to repetition
    num_virtual_tokens=20,
    tokenizer_name_or_path=model_name
)
model = get_peft_model(model, peft_config)

# Metrics with error handling (filter invalid IDs)
bleu = evaluate.load('bleu')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Replace -100 in labels (ignore index) with pad token ID
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Filter out invalid token IDs (e.g., <0 or >vocab_size)
    vocab_size = tokenizer.vocab_size
    predictions = [p for p in predictions if np.all((p >= 0) & (p < vocab_size))]
    labels = [l for l in labels if len(predictions) > 0]  # Align

    if len(predictions) == 0:
        return {"bleu": 0.0}  # Fallback

    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": result["bleu"]}
    except Exception as e:
        print(f"Decode error: {e}")  # Log
        return {"bleu": 0.0}

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Seq2Seq-specific args (better for generation)
training_args = Seq2SeqTrainingArguments(
    output_dir="./p_tuned_query_rewriter",
    num_train_epochs=3,  # Start low for stability
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=True,
    learning_rate=3e-4,  # Even lower for early stability
    warmup_steps=300,  # Extended warmup
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    predict_with_generate=True,
    generation_max_length=128,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to="none",
    # Add gradient clipping to prevent exploding values
    max_grad_norm=1.0,
    # Predict on generation (not logits)
)

# Seq2SeqTrainer (use processing_class)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,  # Or processing_class=tokenizer for future-proof
    compute_metrics=compute_metrics
)

# Train
trainer.train()
trainer.save_model("./fine_tuned_query_rewriter")
tokenizer.save_pretrained("./fine_tuned_query_rewriter")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenized train columns: ['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels']
Sample shape: dict_keys(['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'])


AttributeError: 'T5Config' object has no attribute 'generation_config'

In [ ]:
from datasets import load_dataset
from peft import PromptTuningConfig, get_peft_model, PromptTuningInit, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
import evaluate
import torch
import os
import numpy as np

# Memory env
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# Load dataset
dataset = load_dataset("gaussalgo/Canard_Wiki-augmented")

# Subset for quick testing (before mapping; remove for full)
dataset["train"] = dataset["train"].select(range(1000))  # 1000 train samples
dataset["test"] = dataset["test"].select(range(200))     # 200 eval

# Remove unnecessary original columns before further processing
columns_to_remove = ['History', 'QuAC_dialog_id', 'Question_no', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts']
dataset = dataset.remove_columns(columns_to_remove)


def text_generation(example):
    input_texts = []
    target_texts = []
    for i in range(len(example['Question'])):
        ques = example['Question'][i]
        input_texts.append(f"Rewrite query: {ques}")
        target_texts.append(example['Rewrite'][i])
    example['input_text'] = input_texts
    example['target_text'] = target_texts
    return example

# Format (keeps all columns for now, let remove_unused_columns handle)
dataset = dataset.map(text_generation, batched=True, remove_columns=['Question', 'Rewrite'])


model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16
)

def tokenize(example):
    # Tokenizer expects lists of strings when batched=True
    model_inputs = tokenizer(
        example['input_text'],
        max_length=256,
        truncation=True,
        padding=False  # No padding here – collator handles
    )
    labels = tokenizer(
        example['target_text'],
        max_length=128,
        truncation=True,
        padding=False
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Tokenize with batched=True
tokenized_datasets = dataset.map(tokenize, batched=True)

# Optional: Safe column filter (print to debug)
print("Tokenized train columns:", tokenized_datasets["train"].column_names)  # Should include input_ids, attention_mask, labels
print("Sample shape:", tokenized_datasets["train"][0].keys())

# Keep only essentials if needed (but avoid if causing empty)
# tokenized_datasets = tokenized_datasets.remove_columns([
#     col for col in tokenized_datasets["train"].column_names
#     if col not in ['input_ids', 'attention_mask', 'labels']
# ])

# P-Tuning (RANDOM to minimize echo)
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=20,
    tokenizer_name_or_path=model_name
)
model = get_peft_model(model, peft_config)

# Metrics with safeguards
bleu = evaluate.load('bleu')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Handle -100 in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Filter invalid IDs
    vocab_size = tokenizer.vocab_size
    valid_mask = np.all((predictions >= 0) & (predictions < vocab_size), axis=-1)
    predictions = predictions[valid_mask]
    labels = labels[valid_mask]

    if len(predictions) == 0:
        return {"bleu": 0.0}

    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": result["bleu"]}
    except Exception as e:
        print(f"Metrics error: {e}")
        return {"bleu": 0.0}

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Seq2Seq-specific args (better for generation)
training_args = Seq2SeqTrainingArguments(
    output_dir="./p_tuned_query_rewriter",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=True,
    learning_rate=3e-4,
    warmup_steps=300,
    weight_decay=0.01,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none"
)

# Trainer (use processing_class to avoid warning)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,  # Or processing_class=tokenizer for future-proof
    compute_metrics=compute_metrics
)

# Train and save
trainer.train()
trainer.save_model("./fine_tuned_query_rewriter")
tokenizer.save_pretrained("./fine_tuned_query_rewriter")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenized train columns: ['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels']
Sample shape: dict_keys(['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'])


/tmp/ipython-input-2844878837.py:153: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`History` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [ ]:
peft_config = LoraConfig(
    r=16,  # Low rank
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'], # Add target_modules
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    #dataset_text_field="text",  # Points to your formatted text
    #max_seq_length=MAX_LENGTH,
    #tokenizer=tokenizer,
    args=SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    #max_seq_length=1024,  # truncate long text
    #dataset_text_field="text",
    packing=True,        # True = multiple samples in one sequence
    bf16=True,
    fp16=False
)
    #packing=True,  # TRL bonus: packs multiple examples efficiently
)

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PromptTuningInit, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig  # 🔥 NEW IMPORTS
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,Seq2SeqTrainingArguments,Seq2SeqTrainer, DataCollatorForLanguageModeling
import torch
import evaluate
import numpy as np
import os

# 🔥 OOM FIXES
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# Dataset (SIMPLIFIED - just 'text' column)
dataset = load_dataset("gaussalgo/Canard_Wiki-augmented", split="train[:1000]")
dataset = dataset.train_test_split(test_size=0.1)

def text_generation(example):
    texts = []
    for i in range(len(example['History'])):
        hist = example['History'][i]
        hist_text = "\n".join(str(x) for x in hist) if len(hist) != 0 else 'NO HISTORY'
        ques = example['Question'][i]
        texts.append(f"Rewrite query: {ques}\nResponse: {example['Rewrite'][i]}")
    example['text'] = texts  # ← KEY: 'text' not 'input_text'
    return example

dataset_re = dataset.map(
    text_generation,
    batched=True,
    remove_columns=['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts']
)

# Explicitly remove all columns except 'text'
dataset_re = dataset_re.remove_columns([col for col in dataset_re['train'].column_names if col != 'text'])


# Model & Tokenizer
model_name = "LiquidAI/LFM2-350M-PII-Extract-JP"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map='auto'
)

# 🔥 NO TOKENIZE NEEDED! SFTTrainer handles it

# PEFT Config (unchanged)
peft_config = LoraConfig(
    r=16,  # Low rank
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'], # Add target_modules
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)


model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# 🔥 SFTConfig (Replaces TrainingArguments)
sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    #gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_ratio=0.05,
    logging_steps=10,
    #save_strategy="epoch",
    #max_seq_length=1024,  # truncate long text
    #dataset_text_field="text",
    packing=True,        # True = multiple samples in one sequence
    bf16=True,
    fp16=False
)

# 🔥 BLEU Metrics (SAME)
bleu = evaluate.load('bleu')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[l for l in decoded_labels])
    return {"bleu": result["bleu"]}
datacollector=DataCollatorForLanguageModeling(tokenizer, mlm=False)
# 🔥 SFTTrainer (Replaces Trainer)
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset_re["train"],
    eval_dataset=dataset_re["test"],
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    peft_config=peft_config,
)

# 🔥 RUN - SIMPLER!
sft_trainer.train()
sft_trainer.save_model()

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [ ]:
from datasets import load_dataset
from peft import PromptTuningConfig, get_peft_model, PromptTuningInit, TaskType, prepare_model_for_kbit_training
from transformers import AutoTokenizer, AutoModelForCausalLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, BitsAndBytesConfig, DataCollatorForLanguageModeling
import torch
import evaluate
import numpy as np

# Load & prepare dataset
dataset = load_dataset("gaussalgo/Canard_Wiki-augmented", split="train[:1000]")
dataset = dataset.train_test_split(test_size=0.1)

def text_generation(example):
    input_texts = []
    for i in range(len(example['History'])):
        hist = example['History'][i]
        hist_text = "\n".join(str(x) for x in hist) if len(hist) != 0 else 'NO HISTORY'
        ques = example['Question'][i]
        # Full text: "Rewrite query: {ques} \nResponse: {rewrite}"
        input_texts.append(f"Rewrite query: {ques} \nResponse: {example['Rewrite'][i]}")
    example['input_text'] = input_texts
    return example

dataset_re = dataset.map(text_generation, batched=True, remove_columns=['History', 'QuAC_dialog_id', 'Question', 'Question_no', 'Rewrite', 'true_page_title', 'true_contexts', 'answer', 'true_contexts_wiki', 'extractive', 'retrieved_contexts'])

# FIXED TOKENIZER FUNCTION
def tokenize(example):
    model_inputs = tokenizer(
        example['input_text'],
        max_length=256,
        padding_side='left',
        truncation=True,
        padding='max_length'
    )
    # APPROACH 2: Copy input_ids to labels (full sequence loss)
    #model_inputs['labels'] = model_inputs['input_ids'].copy()
    return model_inputs

# Quantization & Model
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_name = "LiquidAI/LFM2-350M-PII-Extract-JP"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # FIX: Set before model load
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map='auto'
)

# Tokenize dataset
dataset_ready = dataset_re.map(
    tokenize,
    batched=True,
    remove_columns=['input_text']
)

# PEFT Config
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=30,
    tokenizer_name_or_path=model_name
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# Metrics
bleu = evaluate.load('bleu')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Replace -100 with pad_token_id (though none exist in Approach 2)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["bleu"]}

# Training Args
training_args = Seq2SeqTrainingArguments(
    output_dir="./p_tuned_query_rewriter",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    fp16=True,
    learning_rate=5e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    predict_with_generate=True
)
datacollector=DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_ready["train"],
    eval_dataset=dataset_ready["test"],
    compute_metrics=compute_metrics,
    data_collator=datacollector
)

# TRAIN!
trainer.train()
trainer.save_model()

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Bleu
1,No log,6.461605,0.563883


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2066: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


KeyboardInterrupt: 